In [70]:
import pandas as pd

In [71]:
train_df = pd.read_csv('/Users/priyanshu.tuli/Desktop/machinehack/emotions_prediction/Dataset/train.csv')

In [72]:
train_df.head()

,Image_name,Emotion
0,63119.png,happy
1,61769.png,neutral
2,95472.png,disgust
3,93515.png,neutral
4,56585.png,angry


In [73]:
train_df.shape

(7102, 2)

In [74]:
test_df = pd.read_csv('/Users/priyanshu.tuli/Desktop/machinehack/emotions_prediction/Dataset/test.csv')

In [75]:
test_df.head()

,Image_name
0,84757.png
1,57211.png
2,28038.png
3,16408.png
4,43196.png


In [76]:
test_df.shape

(3045, 1)

In [77]:
train_df['Emotion'].value_counts()

Emotion
happy        1008
neutral       899
disregard     881
fear          871
surprise      871
sorrow        871
angry         851
disgust       850
Name: count, dtype: int64

In [78]:
from PIL import Image

In [79]:
from tqdm import tqdm
tqdm.pandas()

In [80]:
import os

In [81]:
for i, row in tqdm(train_df.iterrows()):
    try:
        path = '/Users/priyanshu.tuli/Desktop/machinehack/emotions_prediction/Dataset/Images/' + row['Image_name']
        if not os.path.exists(path):
            path = None
        try:
            img = Image.open(path)
        except:
            path = None
        train_df.at[i, 'image_path'] = path
    except:
        train_df.at[i, 'image_path'] = None
    

7102it [00:01, 3552.43it/s]


In [82]:
for i, row in tqdm(test_df.iterrows()):
    try:
        path = '/Users/priyanshu.tuli/Desktop/machinehack/emotions_prediction/Dataset/Images/'+ row['Image_name']
        if not os.path.exists(path):
            path = None
        try:
            img = Image.open(path)
        except:
            path = None
        test_df.at[i, 'image_path'] = path
    except:
        test_df.at[i, 'image_path'] = None

3045it [00:00, 3641.47it/s]


In [83]:
train_df['image_path'].isna().mean() * 100

0.014080540692762602

In [84]:
test_df['image_path'].isna().mean() * 100

0.0

In [85]:
train_df = train_df.dropna(subset=['image_path'])

In [86]:
train_df.shape

(7101, 3)

In [87]:
train_labels = train_df['Emotion'].unique()

In [88]:
label2idx = {label: idx for idx, label in enumerate(train_labels)}
idx2label = {idx: label for idx, label in enumerate(train_labels)}

In [89]:
from transformers import AutoImageProcessor

In [90]:
checkpoint = "google/vit-base-patch16-224-in21k"

In [91]:
image_processor = AutoImageProcessor.from_pretrained(checkpoint, use_fast=True)

In [92]:
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

In [93]:
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])

In [94]:
def transforms(image):
    return _transforms(image)

In [95]:
import torch

In [96]:
from torch.utils.data import DataLoader, Dataset

In [97]:
class ImageClassificationDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]["image_path"]
        label = self.dataframe.iloc[idx]["label"]

        # Load the image
        image = Image.open(img_path).convert("RGB")

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return {"pixel_values": image, "labels": torch.tensor(label, dtype=torch.long)}

In [98]:
train_df.shape

(7101, 3)

In [99]:
train_df['label'] = train_df['Emotion'].map(label2idx)

In [100]:
complete_train_dataset = ImageClassificationDataset(train_df, transform=transforms)

In [101]:
len(train_dataset)

5680

In [102]:
train_dataset, val_dataset = torch.utils.data.random_split(complete_train_dataset, [int(0.8 * len(complete_train_dataset)), len(complete_train_dataset) - int(0.8 * len(complete_train_dataset))])

In [103]:
len(train_dataset), len(val_dataset)

(5680, 1421)

In [104]:
train_df.head()

,Image_name,Emotion,image_path,label
0,63119.png,happy,/Users/priyanshu.tuli/Desktop/machinehack/emot...,0
1,61769.png,neutral,/Users/priyanshu.tuli/Desktop/machinehack/emot...,1
2,95472.png,disgust,/Users/priyanshu.tuli/Desktop/machinehack/emot...,2
3,93515.png,neutral,/Users/priyanshu.tuli/Desktop/machinehack/emot...,1
4,56585.png,angry,/Users/priyanshu.tuli/Desktop/machinehack/emot...,3


In [105]:
from transformers import DefaultDataCollator

In [106]:
data_collator = DefaultDataCollator()

In [107]:
import evaluate

In [108]:
accuracy = evaluate.load("accuracy")

In [109]:
import numpy as np

In [110]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [111]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

In [112]:
model = AutoModelForImageClassification.from_pretrained(checkpoint, num_labels=len(train_labels),
                                                         label2id=label2idx, id2label=idx2label)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [118]:
training_args = TrainingArguments(
    output_dir="./results_complete",
    remove_unused_columns=False,
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    warmup_ratio=0.1,
    logging_steps=10,
    save_total_limit=2,
    logging_strategy="steps",
    metric_for_best_model="accuracy",
    push_to_hub=False,
    dataloader_num_workers=0,
)

In [119]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=complete_train_dataset,
    tokenizer=image_processor,
    compute_metrics=compute_metrics,
)

/var/folders/pw/wmjkx07d5bj7gwx0bm5zkdjm0000gp/T/ipykernel_41286/1055732089.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [120]:
trainer.train()

Step,Training Loss
10,2.079100
20,2.066500
30,2.075600
40,2.059500
50,2.072300
60,2.070200
70,2.041000
80,2.046400
90,2.019300
100,2.012100


TrainOutput(global_step=2215, training_loss=1.3390036828361838, metrics={'train_runtime': 2158.1352, 'train_samples_per_second': 16.452, 'train_steps_per_second': 1.026, 'total_flos': 2.749563644390277e+18, 'train_loss': 1.3390036828361838, 'epoch': 4.998028724303014})

In [121]:
model.eval()

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fe

In [122]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [123]:
def preprocess_image(image_path, transform):
    image = Image.open(image_path).convert("RGB")  # Open image
    tensor_image = transform(image).unsqueeze(0)
    return tensor_image.to(device)

In [124]:
print("Device:", device)

Device: mps


In [125]:
def predict(image_path, transform):
    input_tensor = preprocess_image(image_path, transform)
    with torch.no_grad():
        outputs = model(input_tensor)
        logits = outputs.logits
        predicted_label = torch.argmax(logits, dim=1).item()
    return idx2label[predicted_label]  # Return class name

In [126]:
test_df['predicted_label'] = test_df['image_path'].progress_apply(lambda x: predict(x, transforms))

100%|██████████| 3045/3045 [01:06<00:00, 46.01it/s]


In [127]:
test_df.head()

,Image_name,image_path,predicted_label
0,84757.png,/Users/priyanshu.tuli/Desktop/machinehack/emot...,happy
1,57211.png,/Users/priyanshu.tuli/Desktop/machinehack/emot...,neutral
2,28038.png,/Users/priyanshu.tuli/Desktop/machinehack/emot...,happy
3,16408.png,/Users/priyanshu.tuli/Desktop/machinehack/emot...,happy
4,43196.png,/Users/priyanshu.tuli/Desktop/machinehack/emot...,disgust


In [128]:
test_df.rename(columns={'predicted_label': 'Emotion'}, inplace=True)

In [129]:
test_df['Emotion'].to_csv('/Users/priyanshu.tuli/Desktop/machinehack/emotions_prediction/Dataset/submission.csv', index=False)